# IDX Exchange DA Internship - Weeks Four and Five - Data Cleaning and Preparation

Submit a cleaned, analysis-ready dataset as a CSV, plus a .py script documenting every transformation
made and why. Include before/after row counts, data type confirmations, date consistency flag counts, and
a geographic data quality summary noting any invalid coordinate records.

In [1]:
import pandas as pd
import numpy as np

## Import Datasets

In [2]:
listings = pd.read_csv("listings_with_rates.csv", low_memory=False)
sold = pd.read_csv("sold_with_rates.csv", low_memory=False)

In [3]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 547162 rows, 72 columns
Shape of Sold Dataset: 414054 rows, 75 columns


## Datetime Conversion

In [4]:
# Some columns involve date-related data, but are not in datetime format
# Converting these columns will make them easier to work with

# listings: CloseDate, CloseDate.1, ListingContractDate, ContractStatusChangeDate, PurchaseContractDate
listings["CloseDate"] = pd.to_datetime(listings["CloseDate"])
listings["CloseDate.1"] = pd.to_datetime(listings["CloseDate.1"])
listings["ListingContractDate"] = pd.to_datetime(listings["ListingContractDate"])
listings["ContractStatusChangeDate"] = pd.to_datetime(listings["ContractStatusChangeDate"])
listings["PurchaseContractDate"] = pd.to_datetime(listings["PurchaseContractDate"])

# sold: CloseDate, ListingContractDate, ContractStatusChangeDate, PurchaseContractDate
sold["CloseDate"] = pd.to_datetime(sold["CloseDate"])
sold["ListingContractDate"] = pd.to_datetime(sold["ListingContractDate"])
sold["ContractStatusChangeDate"] = pd.to_datetime(sold["ContractStatusChangeDate"])
sold["PurchaseContractDate"] = pd.to_datetime(sold["PurchaseContractDate"])

## Remove Unnecessary Columns

In [5]:
listings.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'ListAgentFirstName.1', 'AttachedGarageYN', 'ParkingTotal',
       'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName',
       'BuyerOfficeAOR', 'YearBuilt', 'BuyerAgencyCompensationType',
       'StreetNumberNumeric', 'LivingArea.1', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BedroomsTotal', 'ContractStatusChangeDate', 'Longitude.1',
       'C

In [6]:
sold.columns

Index(['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BedroomsTotal', 'ContractStatusChangeDa

### Dropping Duplicate Columns

In [7]:
# Some columns in the listings dataset have a .1 at the end.
# Check how many rows match each other between the apparent duplicates

full = listings.shape[0]

matching = listings["ListAgentFirstName"].eq(listings["ListAgentFirstName.1"]).sum()
print(f"Difference between ListAgentFirstName: {full - matching}")

matching = listings["LivingArea"].eq(listings["LivingArea.1"]).sum()
print(f"Difference between LivingArea: {full - matching}")

matching = listings["Longitude"].eq(listings["Longitude.1"]).sum()
print(f"Difference between Longitude: {full - matching}")

matching = listings["Latitude"].eq(listings["Latitude.1"]).sum()
print(f"Difference between Latitude: {full - matching}")

matching = listings["CloseDate"].eq(listings["CloseDate.1"]).sum()
print(f"Difference between CloseDate: {full - matching}")

matching = listings["UnparsedAddress"].eq(listings["UnparsedAddress.1"]).sum()
print(f"Difference between UnparsedAddress: {full - matching}")

matching = listings["ListAgentLastName"].eq(listings["ListAgentLastName.1"]).sum()
print(f"Difference between ListAgentLastName: {full - matching}")

matching = listings["BuyerOfficeName"].eq(listings["BuyerOfficeName.1"]).sum()
print(f"Difference between BuyerOfficeName: {full - matching}")

Difference between ListAgentFirstName: 4354
Difference between LivingArea: 552
Difference between Longitude: 80247
Difference between Latitude: 80247
Difference between CloseDate: 388096
Difference between UnparsedAddress: 783
Difference between ListAgentLastName: 42
Difference between BuyerOfficeName: 397276


In [8]:
# The count of non-equal rows matches up (almost) exactly with the amount of nulls rows in each column
# Since .eq counts nulls as False, we can conclude that the .1 columns are exact duplicates

listings[["ListAgentFirstName", "LivingArea", "Longitude", "Latitude", "CloseDate", "UnparsedAddress",
         "ListAgentLastName", "BuyerOfficeName"]].isna().sum()

ListAgentFirstName      4354
LivingArea               552
Longitude              80247
Latitude               80247
CloseDate             388096
UnparsedAddress          782
ListAgentLastName         42
BuyerOfficeName       397275
dtype: int64

In [9]:
# Drop the .1 columns for being exact duplicates

listings = listings.drop(columns = ["ListAgentFirstName.1", "LivingArea.1", "Longitude.1", "Latitude.1", "CloseDate.1",
                          "UnparsedAddress.1", "ListAgentLastName.1", "BuyerOfficeName.1"])

In [10]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 547162 rows, 64 columns
Shape of Sold Dataset: 414054 rows, 75 columns


### Removing Irrelevant Information

In [11]:
# Can delete the ListingKeyNumeric column for being the same as ListingKey

full_1 = listings.shape[0]
full_2 = sold.shape[0]

matching_1 = listings["ListingKey"].eq(listings["ListingKeyNumeric"]).sum()
print(f"Difference between listings ListingKey: {full_1 - matching_1}")

matching_2 = sold["ListingKey"].eq(sold["ListingKeyNumeric"]).sum()
print(f"Difference between sold ListingKey: {full_2 - matching_2}")

listings = listings.drop(columns = ["ListingKeyNumeric"])
sold = sold.drop(columns = ["ListingKeyNumeric"])

Difference between listings ListingKey: 0
Difference between sold ListingKey: 0


In [12]:
# Delete ListAgentFirstName and ListAgentLastName for being redundant with ListAgentFullName

print(listings[["ListAgentFullName", "ListAgentFirstName", "ListAgentLastName"]].isna().sum())
print("-----")
print(sold[["ListAgentFullName", "ListAgentFirstName", "ListAgentLastName"]].isna().sum())

listings = listings.drop(columns = ["ListAgentFirstName", "ListAgentLastName"])
sold = sold.drop(columns = ["ListAgentFirstName", "ListAgentLastName"])

ListAgentFullName       92
ListAgentFirstName    4354
ListAgentLastName       42
dtype: int64
-----
ListAgentFullName       92
ListAgentFirstName    3109
ListAgentLastName       42
dtype: int64


In [13]:
# Combine the other FirstName/LastName columns to match ListAgentFullName

listings["CoListAgentFullName"] = listings["CoListAgentFirstName"] + " " + listings["CoListAgentLastName"]

sold["CoListAgentFullName"] = sold["CoListAgentFirstName"] + " " + sold["CoListAgentLastName"]
sold["BuyerAgentFullName"] = sold["BuyerAgentFirstName"] + " " + sold["BuyerAgentLastName"]

listings = listings.drop(columns = ["CoListAgentFirstName", "CoListAgentLastName"])
sold = sold.drop(columns = ["CoListAgentFirstName", "CoListAgentLastName"])
sold = sold.drop(columns = ["BuyerAgentFirstName", "BuyerAgentLastName"])

In [14]:
# Check null amounts of potentially unncessary columns

print(listings[["BuyerAgencyCompensationType", "BuyerAgencyCompensation",
          "AssociationFee", "AssociationFeeFrequency",
          "ListAgentEmail",
          "CoBuyerAgentFirstName",
          "BuyerAgentMlsId", "BuyerAgentFirstName", "BuyerAgentLastName",
          "BuyerOfficeAOR",
          "BuilderName"]].isna().sum())

print("-----")

print(sold[["BuyerAgencyCompensationType", "BuyerAgencyCompensation",
      "AssociationFee", "AssociationFeeFrequency",
      "ListAgentEmail",
      "CoBuyerAgentFirstName",
      "BuyerAgentMlsId",
      "BuyerOfficeAOR", 'ListAgentAOR', 'BuyerAgentAOR',
      "BuilderName",
      "OriginatingSystemName", "OriginatingSystemSubName"]].isna().sum())

BuyerAgencyCompensationType    468149
BuyerAgencyCompensation        468169
AssociationFee                 132423
AssociationFeeFrequency        313802
ListAgentEmail                   1506
CoBuyerAgentFirstName          533466
BuyerAgentMlsId                395246
BuyerAgentFirstName            395832
BuyerAgentLastName             395138
BuyerOfficeAOR                 406788
BuilderName                    521800
dtype: int64
-----
BuyerAgencyCompensationType    356726
BuyerAgencyCompensation        356739
AssociationFee                  94697
AssociationFeeFrequency        241538
ListAgentEmail                   1065
CoBuyerAgentFirstName          376429
BuyerAgentMlsId                   534
BuyerOfficeAOR                  23396
ListAgentAOR                    57379
BuyerAgentAOR                   62072
BuilderName                    393790
OriginatingSystemName          369545
OriginatingSystemSubName       369545
dtype: int64


In [15]:
# Drop unnecessary columns

### Reasoning:
# BuyerAgencyCompensation info: very high null counts (468,000+ null rows), not very relevant to analyzing success of agency.
# AssocationFee info: fee paid by homeowners to HOA or other organization. Not very relevant to property sales.
# ListAgentEmail: ListAgentFullName is enough to encode list agent information.
# CoBuyerAgentFirstName: only having the first name is not very helpful.
# BuyerAgent info: don't plan to do any analysis with the buyer agents. Also, high missing value in all columns (395,000+)
# BuyerOfficeAOR: very high null values (406,000+), and non-important information.
# BuilderName: not relevant for later analysis, and extremely high null values (521,800+)

listings = listings.drop(columns = ["BuyerAgencyCompensationType", "BuyerAgencyCompensation", 
                                    "AssociationFee", "AssociationFeeFrequency",
                                    "ListAgentEmail", 
                                   "CoBuyerAgentFirstName",
                                   "BuyerAgentMlsId", "BuyerAgentFirstName", "BuyerAgentLastName",
                                   "BuyerOfficeAOR",
                                   "BuilderName"])

### Reasoning:
# BuyerAgencyCompensation info: very high null counts (356,000+ null rows), not very relevant to analyzing success of agency.
# AssocationFee info: fee paid by homeowners to HOA or other organization. Not very relevant to property sales.
# ListAgentEmail: ListAgentFullName is enough to encode list agent information.
# CoBuyerAgentFirstName: only having the first name is not very helpful.
# AOR information: analysis will be limited to individual companies, not bigger AORs.
# BuilderName: not relevant for later analysis, and extremely high null values (393,000+)
# OriginatingSystem: non-relevant metadata.
# lat/lon-filled: non-relevant metadata.

sold = sold.drop(columns = ["BuyerAgencyCompensationType", "BuyerAgencyCompensation", 
                                    "AssociationFee", "AssociationFeeFrequency",
                                    "ListAgentEmail", 
                                   "CoBuyerAgentFirstName",
                                   "BuyerOfficeAOR", "ListAgentAOR", "BuyerAgentAOR",
                                   "BuilderName",
                                   "OriginatingSystemName", "OriginatingSystemSubName",
                                    "latfilled", "lonfilled"])

In [16]:
# Both datasets was previously filtered down to only residential properties.
# Since PropertyType will now always be "Residential" we can get rid of the PropertyType column

print(listings["PropertyType"].unique())
print(sold["PropertyType"].unique())

listings = listings.drop(columns = ["PropertyType"])
sold = sold.drop(columns = ["PropertyType"])

['Residential']
['Residential']


In [17]:
# Sold dataset will only ever have "closed" status, since all properties recorded in it are sold.
# Can drop MlsStatus column since it doesn't tell us anything new

print(sold["MlsStatus"].unique())

sold = sold.drop(columns = ["MlsStatus"])

['Closed']


In [18]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 547162 rows, 48 columns
Shape of Sold Dataset: 414054 rows, 54 columns


## Check Data Consistency

### Invalid Numeric Columns

In [19]:
# Remove rows with ClosePrice <= 0

print(listings[listings["ClosePrice"] <= 0].shape) # 0
print(sold[sold["ClosePrice"] <= 0].shape) # 1

listings = listings[~(listings["ClosePrice"] <= 0)]
sold = sold[~(sold["ClosePrice"] <= 0)]

(0, 48)
(1, 54)


In [20]:
# Remove rows with LivingArea <= 0

print(listings[listings["LivingArea"] <= 0].shape) # 62
print(sold[sold["LivingArea"] <= 0].shape) # 150

listings = listings[~(listings["LivingArea"] <= 0)]
sold = sold[~(sold["LivingArea"] <= 0)]

(360, 48)
(150, 54)


In [21]:
# Remove rows with DaysOnMarket < 0

print(listings[listings["DaysOnMarket"] < 0].shape) # 17
print(sold[sold["DaysOnMarket"] < 0].shape) # 48

listings = listings[~(listings["DaysOnMarket"] < 0)]
sold = sold[~(sold["DaysOnMarket"] < 0)]

(27, 48)
(48, 54)


In [22]:
# Remove rows with negative bathrooms or bedrooms
# Note: no need, there are zero rows with negative bathrooms/bedrooms

print(listings[listings["BathroomsTotalInteger"] < 0].shape) # 0
print(sold[sold["BathroomsTotalInteger"] < 0].shape) # 0

print(listings[listings["BedroomsTotal"] < 0].shape) # 0
print(sold[sold["BedroomsTotal"] < 0].shape) # 0

print(listings[listings["MainLevelBedrooms"] < 0].shape) # 0
print(sold[sold["MainLevelBedrooms"] < 0].shape) # 0

(0, 48)
(0, 54)
(0, 48)
(0, 54)
(0, 48)
(0, 54)


In [23]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546775 rows, 48 columns
Shape of Sold Dataset: 413855 rows, 54 columns


### Invalid Date Columns

In [24]:
# Make sure ListingContractDate is before PurchaseContractDate and 

# How many ListingContractDate come after PurchaseContractDate (invalid)
print(listings[listings["ListingContractDate"] > listings["PurchaseContractDate"]].shape) #212
print(sold[sold["ListingContractDate"] > sold["PurchaseContractDate"]].shape) #267

listings["negative_timeline_flag"] = listings["ListingContractDate"] > listings["PurchaseContractDate"]
sold["negative_timeline_flag"] = sold["ListingContractDate"] > sold["PurchaseContractDate"]

(268, 48)
(267, 54)


In [25]:
# Make sure ListingContractDate is before CloseDate

# How many ListingContractDate come after CloseDate (invalid)
print(listings[listings["ListingContractDate"] > listings["CloseDate"]].shape) #72
print(sold[sold["ListingContractDate"] > sold["CloseDate"]].shape) #61

listings["listing_after_close_flag"] = listings["ListingContractDate"] > listings["CloseDate"]
sold["listing_after_close_flag"] = sold["ListingContractDate"] > sold["CloseDate"]

(73, 49)
(61, 55)


In [26]:
# Make sure PurchaseContractDate is before CloseDate

# How many PurchaseContractDate come after CloseDate (invalid)
print(listings[listings["PurchaseContractDate"] > listings["CloseDate"]].shape) #245
print(sold[sold["PurchaseContractDate"] > sold["CloseDate"]].shape) #239

listings["purchase_after_close_flag"] = listings["PurchaseContractDate"] > listings["CloseDate"]
sold["purchase_after_close_flag"] = sold["PurchaseContractDate"] > sold["CloseDate"]

(262, 50)
(239, 56)


In [27]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546775 rows, 51 columns
Shape of Sold Dataset: 413855 rows, 57 columns


### Invalid Geographic Columns

In [28]:
# California coordinates will be negative. Remove any that are not California based
# Other coordianted are handled in in the "Location Features" section

print(listings[listings["Longitude"] >= 0].shape) #137
print(sold[sold["Longitude"] >= 0].shape) #58

listings = listings[~(listings["Longitude"] >= 0)]
sold = sold[~(sold["Longitude"] >= 0)]

(137, 51)
(58, 57)


In [29]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413797 rows, 57 columns


## Handle Missing Values

In [30]:
# Create datasets to investigate missing values

listings_na = (listings.isna().mean() * 100).sort_values(ascending=False).to_frame(name="Percent")
listings_na["Counts"] = listings.isna().sum()

sold_na = (sold.isna().mean() * 100).sort_values(ascending=False).to_frame(name="Percent")
sold_na["Count"] = sold.isna().sum()

### Keys

In [31]:
# Check ListingKey, ListingId

# Both have no missing values

print(listings_na.loc[["ListingKey", "ListingId"]])
print(sold_na.loc[["ListingKey", "ListingId"]])

            Percent  Counts
ListingKey      0.0       0
ListingId       0.0       0
            Percent  Count
ListingKey      0.0      0
ListingId       0.0      0


In [32]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413797 rows, 57 columns


### PropertyType

In [33]:
# Check PropertySubType, MlsStatus (listings)
# Check PropertySubType (sold)

# MlsStatus has no null values, but PropertySubType does for both datasets

print(listings_na.loc[["PropertySubType", "MlsStatus"]])
print(sold_na.loc[["PropertySubType"]])

                  Percent  Counts
PropertySubType  0.230317    1259
MlsStatus        0.000000       0
                  Percent  Count
PropertySubType  0.194298    804


In [34]:
# Fill PropertySubType with "Unknown"

listings["PropertySubType"] = listings["PropertySubType"].fillna("Unknown")
sold["PropertySubType"] = sold["PropertySubType"].fillna("Unknown")

In [35]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413797 rows, 57 columns


### Price Features

In [36]:
# Check OriginalListPrice, ListPrice, ClosePrice, Rate_30yr_fixed

# ListPrice and rate_30yr_fixed are full, but OriginalListPrice and ClosePrice need to be filled

print(listings_na.loc[["OriginalListPrice", "ListPrice", "ClosePrice", "rate_30yr_fixed"]])
print(sold_na.loc[["OriginalListPrice", "ListPrice", "ClosePrice", "rate_30yr_fixed"]])

                     Percent  Counts
OriginalListPrice   0.144337     789
ListPrice           0.000000       0
ClosePrice         74.830144  409050
rate_30yr_fixed     0.000000       0
                    Percent  Count
OriginalListPrice  0.182698    756
ListPrice          0.000000      0
ClosePrice         0.000483      2
rate_30yr_fixed    0.000000      0


In [37]:
# Impute OriginalListPrice with ListPrice if it's missing

listings["OriginalListPrice"] = listings["OriginalListPrice"].fillna(listings["ListPrice"])
sold["OriginalListPrice"] = sold["OriginalListPrice"].fillna(sold["ListPrice"])

In [38]:
# Listings has so many missing ClosePrice values because many properties are not yet closed
# Fill missing ClosePrice values that are not yet closed with "N/A"
# Otherwise, fill with "Unknown"

print(listings.groupby("MlsStatus")["ClosePrice"].count())

listings.loc[(listings["ClosePrice"].isna()) & 
                    (listings["MlsStatus"] == "Active"),
                    "ClosePrice"] = "N/A"

listings.loc[(listings["ClosePrice"].isna()) & 
                    (listings["MlsStatus"] == "ActiveUnderContract"),
                    "ClosePrice"] = "N/A"

listings.loc[(listings["ClosePrice"].isna()) & 
                    (listings["MlsStatus"] == "Closed"),
                    "ClosePrice"] = "Unknown"

listings.loc[(listings["ClosePrice"].isna()) & 
                    (listings["MlsStatus"] == "ComingSoon"),
                    "ClosePrice"] = "N/A"

listings.loc[(listings["ClosePrice"].isna()) & 
                    (listings["MlsStatus"] == "Pending"),
                    "ClosePrice"] = "Unknown"

MlsStatus
Active                      0
ActiveUnderContract         2
Closed                 137583
ComingSoon                  0
Pending                     3
Name: ClosePrice, dtype: int64


/var/folders/p2/8wbydb310gg6z0sdxkw7vhym0000gn/T/ipykernel_25631/3447684796.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  listings.loc[(listings["ClosePrice"].isna()) &


In [39]:
# For sold dataset, only two rows with ClosePrice missing, can drop those

sold = sold.dropna(subset=["ClosePrice"])

In [40]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413795 rows, 57 columns


### Office Features

In [41]:
# Check ListOfficeName, BuyerOfficeName, CoListOfficeName

# ListOfficeName is full, but BuyerOfficeName and CoListOfficeName need to be filled

print(listings_na.loc[["ListOfficeName", "BuyerOfficeName", "CoListOfficeName"]])
print(sold_na.loc[["ListOfficeName", "BuyerOfficeName", "CoListOfficeName"]])

                    Percent  Counts
ListOfficeName     0.000000       0
BuyerOfficeName   72.598136  396849
CoListOfficeName  77.640230  424411
                    Percent   Count
ListOfficeName     0.000000       0
BuyerOfficeName    1.628818    6740
CoListOfficeName  76.333806  315867


In [42]:
# Most missing BuyerOfficeName in listings dataset is because the property hasn't been bought yet
# Fill missing ClosePrice values that are not yet closed with "N/A"
# Otherwise, fill with "Unknown"

print(listings.groupby("MlsStatus")["BuyerOfficeName"].count())

listings.loc[(listings["BuyerOfficeName"].isna()) & 
                    (listings["MlsStatus"] == "Active"),
                    "BuyerOfficeName"] = "N/A"

listings.loc[(listings["BuyerOfficeName"].isna()) & 
                    (listings["MlsStatus"] == "ActiveUnderContract"),
                    "BuyerOfficeName"] = "N/A"

listings.loc[(listings["BuyerOfficeName"].isna()) & 
                    (listings["MlsStatus"] == "Closed"),
                    "BuyerOfficeName"] = "Unknown"

listings.loc[(listings["BuyerOfficeName"].isna()) & 
                    (listings["MlsStatus"] == "ComingSoon"),
                    "BuyerOfficeName"] = "N/A"

listings.loc[(listings["BuyerOfficeName"].isna()) & 
                    (listings["MlsStatus"] == "Pending"),
                    "BuyerOfficeName"] = "Unknown"

MlsStatus
Active                     11
ActiveUnderContract         6
Closed                 135373
ComingSoon                  0
Pending                 14399
Name: BuyerOfficeName, dtype: int64


In [43]:
# For sold dataset, fill BuyerOfficeName with "Unknown"

sold["BuyerOfficeName"] = sold["BuyerOfficeName"].fillna("Unknown")

In [44]:
# For both listings and sold datatset, fill CoListOfficeName with "Unknown"

listings["CoListOfficeName"] = listings["CoListOfficeName"].fillna("Unknown")
sold["CoListOfficeName"] = sold["CoListOfficeName"].fillna("Unknown")

In [45]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413795 rows, 57 columns


### Agent Features

In [46]:
# Check ListAgentFullName, CoListAgentFullName (listings)
# Check ListAgentFullName, CoListAgentFullName, BuyerAgentFullName, BuyerAgentMlsId (sold)

print(listings_na.loc[["ListAgentFullName", "CoListAgentFullName"]])
print(sold_na.loc[["ListAgentFullName", "CoListAgentFullName", "BuyerAgentFullName", "BuyerAgentMlsId"]])

                       Percent  Counts
ListAgentFullName     0.016830      92
CoListAgentFullName  77.732064  424913
                       Percent   Count
ListAgentFullName     0.022233      92
CoListAgentFullName  77.083691  318970
BuyerAgentFullName    0.491062    2032
BuyerAgentMlsId       0.128807     533


In [47]:
# Fill all agent name fields with "Unknown"

listings["ListAgentFullName"] = listings["ListAgentFullName"].fillna("Unknown")
listings["CoListAgentFullName"] = listings["CoListAgentFullName"].fillna("Unknown")

sold["ListAgentFullName"] = sold["ListAgentFullName"].fillna("Unknown")
sold["CoListAgentFullName"] = sold["CoListAgentFullName"].fillna("Unknown")
sold["BuyerAgentFullName"] = sold["BuyerAgentFullName"].fillna("Unknown")
sold["BuyerAgentMlsId"] = sold["BuyerAgentMlsId"].fillna("Unknown")

In [48]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413795 rows, 57 columns


### Time Features

In [49]:
# Check CloseDate, DaysOnMarket, YearBuilt, ContractStatusChangeDate, PurchaseContractDate, ListingContractDate, year_month

# Listings: DaysOnMarket, ListingContractDate, year_month are ok. Investigate CloseDate, YearBuilt, ContractStatusChangeDate, PurchaseContractDate
# Sold: CloseDate, DaysOnMarket, year_month are ok. Investigate YearBuilt, ContractStatusChangeDate, PurchaseContractDate, ListingContractDate

print(listings_na.loc[["CloseDate", "DaysOnMarket", "YearBuilt", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate", "year_month"]])
print(sold_na.loc[["CloseDate", "DaysOnMarket", "YearBuilt", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate", "year_month"]])

                            Percent  Counts
CloseDate                 70.920609  387679
DaysOnMarket               0.000000       0
YearBuilt                  0.158240     865
ContractStatusChangeDate   1.153597    6306
PurchaseContractDate      52.193408  285309
ListingContractDate        0.000000       0
year_month                 0.000000       0
                           Percent  Count
CloseDate                 0.000000      0
DaysOnMarket              0.000000      0
YearBuilt                 0.083616    346
ContractStatusChangeDate  0.143549    594
PurchaseContractDate      0.046883    194
ListingContractDate       0.000242      1
year_month                0.000000      0


In [50]:
# Most missing CloseDates in listings are because the properties haven't been closed yet
# Fill missing CloseDate values that are not yet closed, fill with "N/A"
# Otherwise, fill with "Unknown"

print(listings.groupby("MlsStatus")["CloseDate"].count())

listings.loc[(listings["CloseDate"].isna()) & 
                    (listings["MlsStatus"] == "Active"),
                    "CloseDate"] = "N/A"

listings.loc[(listings["CloseDate"].isna()) & 
                    (listings["MlsStatus"] == "ActiveUnderContract"),
                    "CloseDate"] = "N/A"

listings.loc[(listings["CloseDate"].isna()) & 
                    (listings["MlsStatus"] == "Closed"),
                    "CloseDate"] = "Unknown"

listings.loc[(listings["CloseDate"].isna()) & 
                    (listings["MlsStatus"] == "ComingSoon"),
                    "CloseDate"] = "N/A"

listings.loc[(listings["CloseDate"].isna()) & 
                    (listings["MlsStatus"] == "Pending"),
                    "CloseDate"] = "Unknown"

MlsStatus
Active                      7
ActiveUnderContract      3671
Closed                 137583
ComingSoon                  0
Pending                 17698
Name: CloseDate, dtype: int64


/var/folders/p2/8wbydb310gg6z0sdxkw7vhym0000gn/T/ipykernel_25631/2142945494.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  listings.loc[(listings["CloseDate"].isna()) &


In [51]:
# Both datasets have a few missing YearBuilt values
# There's no good way to try to impute this value, so just fill with "Unknown"

listings["YearBuilt"] = listings["YearBuilt"].fillna("Unknown")
sold["YearBuilt"] = sold["YearBuilt"].fillna("Unknown")

In [52]:
# ContractStatusChangeDate doesn't seem to change much with MlsStatus. Fill with "Unknown"

print(listings.groupby("MlsStatus")["ContractStatusChangeDate"].count())

listings["ContractStatusChangeDate"] = listings["ContractStatusChangeDate"].fillna("Unknown")
sold["ContractStatusChangeDate"] = sold["ContractStatusChangeDate"].fillna("Unknown")

MlsStatus
Active                 277341
ActiveUnderContract     36354
Closed                 137568
ComingSoon               1672
Pending                 87397
Name: ContractStatusChangeDate, dtype: int64


In [53]:
# PurchaseContractDate is defined as:
# "with for-sale listings, the date an offer was accepted and the listing was no longer on market"
# Which is why there are a strengely high amount of ActiveUnderContract/Pending values with this filled
# For this value, fill all nulls with "Unknown," since n/a is techically not true in this case

print(listings.groupby("MlsStatus")["PurchaseContractDate"].count())

listings["PurchaseContractDate"] = listings["PurchaseContractDate"].fillna("Unknown")
sold["PurchaseContractDate"] = sold["PurchaseContractDate"].fillna("Unknown")

MlsStatus
Active                     24
ActiveUnderContract     36354
Closed                 137552
ComingSoon                  0
Pending                 87399
Name: PurchaseContractDate, dtype: int64


In [54]:
# Only one missing value for ListingContractDate in the sold dataset. Drop this one row

sold = sold.dropna(subset=["ListingContractDate"])

In [55]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546638 rows, 51 columns
Shape of Sold Dataset: 413794 rows, 57 columns


### Location Features

In [56]:
# Check Latitude, Longitude, UnparsedAddress, StreetNumberNumeric, MLSAreaMajor, SubdivisionName,
# PostalCode, City, CountyOrParish, StateOrProvince

# City (listings), CountyOrParish and StateOrProvince (sold) have no null values.

print(listings_na.loc[["Latitude", "Longitude", "UnparsedAddress", "StreetNumberNumeric", "MLSAreaMajor", "SubdivisionName",
                      "PostalCode", "City", "CountyOrParish", "StateOrProvince"]])
print(sold_na.loc[["Latitude", "Longitude", "UnparsedAddress", "StreetNumberNumeric", "MLSAreaMajor", "SubdivisionName",
                      "PostalCode", "City", "CountyOrParish", "StateOrProvince"]])

                       Percent  Counts
Latitude             14.676074   80225
Longitude            14.676074   80225
UnparsedAddress       0.142873     781
StreetNumberNumeric   0.177082     968
MLSAreaMajor         13.613214   74415
SubdivisionName      63.221547  345593
PostalCode            0.002378      13
City                  0.101347     554
CountyOrParish        0.000000       0
StateOrProvince       0.005671      31
                       Percent   Count
Latitude              0.991307    4102
Longitude             0.991307    4102
UnparsedAddress       0.094249     390
StreetNumberNumeric   0.115274     477
MLSAreaMajor         13.371291   55330
SubdivisionName      62.803742  259880
PostalCode            0.000483       2
City                  0.074916     310
CountyOrParish        0.000000       0
StateOrProvince       0.000000       0


In [57]:
# Drop StreetNumberNumeric. This column doesn't tell us much that UnparsedAddress doesn't

listings = listings.drop(columns = ["StreetNumberNumeric"])
sold = sold.drop(columns = ["StreetNumberNumeric"])

In [58]:
# Fill UnparsedAddress with "Unknown"

listings["UnparsedAddress"] = listings["UnparsedAddress"].fillna("Unknown")
sold["UnparsedAddress"] = sold["UnparsedAddress"].fillna("Unknown")

In [59]:
# Very few rows that don't have a postal code. Drop rows without them

listings = listings.dropna(subset = ["PostalCode"])
sold = sold.dropna(subset = ["PostalCode"])

In [60]:
# SubdivisionName has more than half missing rows, and the information doesn't seem essential

listings = listings.drop(columns = ["SubdivisionName"])
sold = sold.drop(columns = ["SubdivisionName"])

In [61]:
# Fill missing MLSAreaMajor with Unknown.
# Though note that "City" seems to be an ok way to impute MLSAreaMajor
# The only reason for not imputing MLSAreaMajor with City is because MLSAreaMajor has a unique format of
# [ID] - [Area Name]. Imputing with city would miss out on the ID part.

listings["MLSAreaMajor"] = listings["MLSAreaMajor"].fillna("Unknown")
sold["MLSAreaMajor"] = sold["MLSAreaMajor"].fillna("Unknown")

In [62]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 546625 rows, 49 columns
Shape of Sold Dataset: 413792 rows, 55 columns


#### Imputation using PostalCode

In [63]:
!pip install uszipcode -q

In [64]:
pip install --upgrade uszipcode -q

Note: you may need to restart the kernel to use updated packages.


In [65]:
# import uszipcode database (above pip installs are necessary for using it)
# website: https://pypi.org/project/uszipcode/
# Has information on zipcodes
# Based on 2020 census data, however it's not out of date, since I'm only using the locational, not demographic data

# Can impute Latitude, Longitude, City, and StateOrProvince using the uszipcode database

from uszipcode import SearchEngine
search = SearchEngine()

/opt/anaconda3/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [66]:
# Check which zipcodes are not registered in the uszipcode database (and are therefore invalid)

# Listings Dataset
invalid_L = []
zipcodes_L = listings[listings["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in zipcodes_L:
    result_L = search.by_zipcode(i)
    if result_L is None:
        invalid_L.append(i)

# Sold Dataset
invalid_S = []
zipcodes_S = sold[sold["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in zipcodes_S:
    result_S = search.by_zipcode(i)
    if result_S is None:
        invalid_S.append(i)

In [75]:
# Check the list of invalid PostalCodes to see if there's a pattern

print(set(invalid_L))
print(len(invalid_L))

{'94578-1907', '94521-3210', '95354-3302', '94536-7409', '94803-3513', '94596-3358', '94509-3154', '94804-5102', '94587-2116', '91336', '94561-1642', '94582-5667', '95758-5723', '94568-7760', '94546-1915', '94565-7234', '95330-0000', '94555-2240', '94550-7205', '94579-2766', '94564-0000', '94705-1406', '94596-6121', '94545-3404', '94531-9070', '94521-4756', '94546-4031', '94404-2809', '94561-3071', '94609-1626', '94710-2809', '94709-1200', '94560-200', '94116-1717', '94803-3226', '94585-2820', '94519-1634', '94538-100', '94509-6938', '94804-1307', '94521-3845', '94517-1508', '94521-3628', '94518-1240', '94602-1713', '94541-4398', '94549-4506', '94523-4025', '95204-5228', '95132-2118', '95035-6218', '94568-7212', '94611-2916', '94530-2120', '94587-1668', '94568-1379', '63119-2447', '94521-1142', '94608-1651', '94530-3328', '94552-1646', '94709-1491', '95376-4735', '94588-8222', '94549-4630', '94531-9133', '94513-5278', '94588-3921', '94513-2020', '94597-2838', '94611-4325', '94518-3851'

In [427]:
# Check the list of invalid PostalCodes to see if there's a pattern

print(invalid_S)

['94611-4007', '94577-5450', '94596-6444', '94804-1101', '94560-1710', '94582-5069', '94552-5493', '95337-6695', '94611-3402', '94710-2809', '94588-3967', '95206-2027', '94519-1610', '94609-1626', '94609-1626', '94605-3813', '94587-4913', '94588-3111', '94597-2519', '95306-9708', '94587-4748', '94607-4252', '94501-0000', '94611-3819', '94526-4236', '94506-6020', '95336-5956', '95386-9479', '94044-4140', '94549-2925', '94509-6864', '94521-5103', '94603-4111', '94536-4263', '95386-9551', '91748-3992', '94530-2567', '94568-4697', '94561-9999', '94551-7536', '94509-5639', '94520-1605', '94803-3627', '94513-4478', '94607-4617', '94804-4303', '94509-5966', '94513-5168', '94526-6226', '94596-3329', '94513-4255', '94568-2053', '94506-4637', '94513-6479', '94536-5026', '94601-5102', '94590-2400', '95330-8896', '95240-0000', '95330-0000', '94603-2034', '94624-3118', '94505-9480', '94506-1918', '94561-0000', '94601-3843', '94509-8923', '95758-7921', '94531-9317', '95330-0000', '94806-2168', '9512

In [428]:
# Many rows seem to have invalid values where PostalCode is represented as a range. Remove these invalid rows

pattern = r'-'
listings = listings[~listings["PostalCode"].str.contains(pattern, na=False, regex=True)]
sold = sold[~sold["PostalCode"].str.contains(pattern, na=False, regex=True)]

In [429]:
# Recompute the invalid PostalCodes without the invalid range values

# Listings Dataset
sec_invalid_L = []
sec_zipcodes_L = listings[listings["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in sec_zipcodes_L:
    result_L = search.by_zipcode(i)
    if result_L is None:
        sec_invalid_L.append(i)

# Sold Dataset
sec_invalid_S = []
sec_zipcodes_S = sold[sold["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in sec_zipcodes_S:
    result_S = search.by_zipcode(i)
    if result_S is None:
        sec_invalid_S.append(i)

In [430]:
# Look at each invalid PostalCode

unique_invalid_L = list(set(sec_invalid_L))
print(unique_invalid_L)

print("-----")

unique_invalid_S = list(set(sec_invalid_S))
print(unique_invalid_S)

['94838', '93413', '13910', '94840', '94821', '93350', '91727', '84568', '95705', '22634', '90470', '0', '99999', '92565', '95530', '22666', '91336', '91700', '94081', '91713', '92892', '63720', '77760', '00000', '22753', '94322']
-----
['93567', '93350', '49603']


In [431]:
# Investigate each invalid PostalCode
# Many of these seem to be data entry errors. 
# The correct PostalCode (from Google) is written as a comment next to each invalid PostalCode

listings[listings["PostalCode"] == "91336"]["City"] # Encino -> 91316
listings[listings["PostalCode"] == "22666"]["City"] # INVALID
listings[listings["PostalCode"] == "95705"]["City"] # Berkeley -> 94705
listings[listings["PostalCode"] == "94081"]["City"] # Richmond -> 94801
listings[listings["PostalCode"] == "92565"]["City"] # Pittsburg -> 94565
listings[listings["PostalCode"] == "91700"]["City"] # Baldwin Park -> 91706
listings[listings["PostalCode"] == "84568"]["City"] # Dublin -> 94568
listings[listings["PostalCode"] == "93350"]["City"] # Palmdale -> 93550 OR Bakersfield -> 93305
listings[listings["PostalCode"] == "94322"]["City"] # Clearlake -> 95422
listings[listings["PostalCode"] == "99999"]["City"] # INVALID
listings[listings["PostalCode"] == "22634"]["City"] # INVALID
listings[listings["PostalCode"] == "0"]["City"] # INVALID
listings[listings["PostalCode"] == "90470"]["City"] # Seal Beach -> 90740
listings[listings["PostalCode"] == "95530"]["City"] # Lathrop -> 95330
listings[listings["PostalCode"] == "94840"]["City"] # Richmond -> 94804
listings[listings["PostalCode"] == "94838"]["City"] # Sacramento -> 95838
listings[listings["PostalCode"] == "63720"]["City"] # INVALID
listings[listings["PostalCode"] == "94821"]["City"] # Sacramento -> 95821
listings[listings["PostalCode"] == "22753"]["City"] # INVALID
listings[listings["PostalCode"] == "13910"]["City"] # INVALID
listings[listings["PostalCode"] == "91713"]["City"] # Lakewood -> 90713
listings[listings["PostalCode"] == "77760"]["City"] # INVALID
listings[listings["PostalCode"] == "91727"]["City"] # Covina -> 91722
listings[listings["PostalCode"] == "00000"]["City"] # INVALID
listings[listings["PostalCode"] == "93413"]["City"] # Brentwood -> 94513
listings[listings["PostalCode"] == "92892"]["City"] # Anaheim -> 92801

sold[sold["PostalCode"] == "93567"]["City"] # Sanger -> 93657
sold[sold["PostalCode"] == "93350"]["City"] # Palmdale -> 93550 OR Bakersfield -> 93305
sold[sold["PostalCode"] == "49603"]["City"] # Oakland - > 94603

371535    Oakland
Name: City, dtype: object

In [432]:
# Remove all the PostalCodes identified as veing invalid, and replace the incorrect PostalCodes with their correct versions

listings = listings[~listings["PostalCode"].str.contains("22666", na=False)]
listings = listings[~listings["PostalCode"].str.contains("99999", na=False)]
listings = listings[~listings["PostalCode"].str.contains("22634", na=False)]
listings = listings[~listings["PostalCode"].str.contains("0", na=False)]
listings = listings[~listings["PostalCode"].str.contains("63720", na=False)]
listings = listings[~listings["PostalCode"].str.contains("22753", na=False)]
listings = listings[~listings["PostalCode"].str.contains("13910", na=False)]
listings = listings[~listings["PostalCode"].str.contains("77760", na=False)]
listings = listings[~listings["PostalCode"].str.contains("00000", na=False)]

listings.loc[(listings["PostalCode"] == "91336") & (listings["City"] == "Encino"), "PostalCode"] = "91316"
listings.loc[(listings["PostalCode"] == "95705") & (listings["City"] == "Berkeley"), "PostalCode"] = "94705"
listings.loc[(listings["PostalCode"] == "94081") & (listings["City"] == "Richmond"), "PostalCode"] = "94081"
listings.loc[(listings["PostalCode"] == "92565") & (listings["City"] == "Pittsburg"), "PostalCode"] = "94565"
listings.loc[(listings["PostalCode"] == "91700") & (listings["City"] == "Baldwin Park"), "PostalCode"] = "91706"
listings.loc[(listings["PostalCode"] == "84568") & (listings["City"] == "Dublin"), "PostalCode"] = "94568"
listings.loc[(listings["PostalCode"] == "93350") & (listings["City"] == "Palmdale"), "PostalCode"] = "93550"
listings.loc[(listings["PostalCode"] == "93350") & (listings["City"] == "Bakersfield"), "PostalCode"] = "93305"
listings.loc[(listings["PostalCode"] == "94322") & (listings["City"] == "Clearlake"), "PostalCode"] = "95422"
listings.loc[(listings["PostalCode"] == "90470") & (listings["City"] == "Seal Beach"), "PostalCode"] = "90740"
listings.loc[(listings["PostalCode"] == "95530") & (listings["City"] == "Lathrop"), "PostalCode"] = "95330"
listings.loc[(listings["PostalCode"] == "94840") & (listings["City"] == "Richmond"), "PostalCode"] = "94804"
listings.loc[(listings["PostalCode"] == "94838") & (listings["City"] == "Sacramento"), "PostalCode"] = "95838"
listings.loc[(listings["PostalCode"] == "94821") & (listings["City"] == "Sacramento"), "PostalCode"] = "95821"
listings.loc[(listings["PostalCode"] == "91713") & (listings["City"] == "Lakewood"), "PostalCode"] = "90713"
listings.loc[(listings["PostalCode"] == "91727") & (listings["City"] == "Covina"), "PostalCode"] = "91722"
listings.loc[(listings["PostalCode"] == "93413") & (listings["City"] == "Brentwood"), "PostalCode"] = "94513"
listings.loc[(listings["PostalCode"] == "92892") & (listings["City"] == "Anaheim"), "PostalCode"] = "92801"

sold.loc[(sold["PostalCode"] == "93567") & (sold["City"] == "Sanger"), "PostalCode"] = "93657"
sold.loc[(sold["PostalCode"] == "93350") & (sold["City"] == "Palmdale"), "PostalCode"] = "93550"
sold.loc[(sold["PostalCode"] == "93350") & (sold["City"] == "Bakersfield"), "PostalCode"] = "93305"
sold.loc[(sold["PostalCode"] == "49603") & (sold["City"] == "Oakland"), "PostalCode"] = "94603"

In [433]:
# Now that all invalid postcodes have been removed, impute Longitude and Latitude

# For each postalcode, create a dictionary where the postalcode's long/lat are saved

# Listings Dataset
lat_dict_L = {}
long_dict_L = {}
zipcodes_L = listings[listings["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in zipcodes_L:
    result_L = search.by_zipcode(i)
    lat_dict_L[i] = result_L.lat
    long_dict_L[i] = result_L.lng

# Sold Dataset
lat_dict_S = {}
long_dict_S = {}
zipcodes_S = sold[sold["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in zipcodes_S:
    result_S = search.by_zipcode(i)
    lat_dict_S[i] = result_S.lat
    long_dict_S[i] = result_S.lng

In [434]:
# Fill in remaning null with the dictionaries created from uszipcodes

listings["Longitude"] = listings["Longitude"].fillna(listings["PostalCode"].map(long_dict_L))
listings["Latitude"] = listings["Latitude"].fillna(listings["PostalCode"].map(lat_dict_L))

sold["Longitude"] = sold["Longitude"].fillna(sold["PostalCode"].map(long_dict_S))
sold["Latitude"] = sold["Latitude"].fillna(sold["PostalCode"].map(lat_dict_S))

In [435]:
# Do the same process as above, but this time for city.
# For every missing city value, get its assocaited PostalCode and pass that through uszipcode.

# Listings Dataset
city_invalid_L = []
city_zipcodes_L = listings[listings["City"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in city_zipcodes_L:
    result_L = search.by_zipcode(i)
    if result_L is None:
        city_invalid_L.append(i)

# Sold Dataset
city_invalid_S = []
city_zipcodes_S = sold[sold["City"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in city_zipcodes_S:
    result_S = search.by_zipcode(i)
    if result_S is None:
        city_invalid_S.append(i)

In [436]:
print(set(city_invalid_L))
print(set(city_invalid_S))

{'76146'}
{'93506', '92440'}


In [437]:
# Investigate invalid zipcodes 

listings[listings["PostalCode"] == "76146"][["PostalCode", "UnparsedAddress", "Longitude", "Latitude", "City"]] #INVALID

sold[sold["PostalCode"] == "93506"][["PostalCode", "UnparsedAddress", "Longitude", "Latitude", "City"]] #INVALID
sold[sold["PostalCode"] == "92440"][["PostalCode", "UnparsedAddress", "Longitude", "Latitude", "City"]] # Desert Hot Springs -> 92440

,PostalCode,UnparsedAddress,Longitude,Latitude,City
252693,92440,62428 Highwing Street,-116.567859,33.968449,Desert Hot Springs
284303,92440,62447 Highwing Street,-116.566796,33.968075,Desert Hot Springs
284305,92440,62448 Highwing Street,-116.566850,33.968494,Desert Hot Springs
295010,92440,62480 Highwing Street,-116.565670,33.968423,Desert Hot Springs
341439,92440,11378 N Star Lane,-116.565670,33.968423,Desert Hot Springs
343469,92440,11334 North Star Lane,-116.565026,33.968733,Desert Hot Springs
344178,92440,62460 Highwing Street,-116.566324,33.968378,Desert Hot Springs
371835,92440,11390 North Star Lane,-116.564908,33.967852,NaN
376766,92440,11374 North Star Lane,-116.564908,33.967852,Desert Hot Springs
376771,92440,11395 North Star Lane,-116.565670,33.968423,NaN


In [438]:
# Get rid of the invalid PostalCode rows
listings = listings[~listings["PostalCode"].str.contains("76146", na=False)]
sold = sold[~sold["PostalCode"].str.contains("93506", na=False)]

# Impute missing city values of 92240 with Desert Hot Springs
sold.loc[(sold["City"].isna()) & 
        (sold["PostalCode"] == "92440"),
        "City"] = "Desert Hot Springs"

# Then change 92240 values to the correct 92240 zipcode
sold["PostalCode"] = sold["PostalCode"].replace("92440", "92240")

In [439]:
# With invalid PostalCodes removed, find the city from the PostalCode

# Listings Dataset
city_dict_L = {}
city_zipcodes_L = listings[listings["City"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in city_zipcodes_L:
    result_L = search.by_zipcode(i)
    city_dict_L[i] = result_L.major_city

# Sold Dataset
city_dict_S = {}
city_zipcodes_S = sold[sold["City"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in city_zipcodes_S:
    result_S = search.by_zipcode(i)
    city_dict_S[i] = result_S.major_city

In [440]:
# Impute remaining city values with PostalCode-City dictionary

listings["City"] = listings["City"].fillna(listings["PostalCode"].map(city_dict_L))
sold["City"] = sold["City"].fillna(sold["PostalCode"].map(city_dict_S))

In [441]:
# Repeat process for StateOrProvince (listings)

# Check which zipcodes are not registered in the uszipcode database (and are therefore invalid)

# Listings Dataset
invalid_L = []
zipcodes_L = listings[listings["StateOrProvince"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in zipcodes_L:
    result_L = search.by_zipcode(i)
    if result_L is None:
        invalid_L.append(i)

In [442]:
print(invalid_L)

[]


In [443]:
# For each missing StateOrProvince, grab the PostalCode and use uszipcodes to grab the state data

state_dict_L = {}
state_zipcodes_L = listings[listings["StateOrProvince"].isna() == True][["PostalCode"]].values.flatten().tolist()

for i in state_zipcodes_L:
    result_L = search.by_zipcode(i)
    state_dict_L[i] = result_L.state

In [444]:
# Impute missing StateOrProvince values from the PostalCode-State dictionary

listings["StateOrProvince"] = listings["StateOrProvince"].fillna(listings["PostalCode"].map(state_dict_L))

In [445]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 49 columns
Shape of Sold Dataset: 411631 rows, 55 columns


### School Features

In [446]:
# Check ElementarySchool, MiddleOrJuniorSchool, HighSchool, HighSchoolDistrict

# Fairly high missing percentage for each feature

print(listings_na.loc[["ElementarySchool", "MiddleOrJuniorSchool", "HighSchool", "HighSchoolDistrict"]])
print(sold_na.loc[["ElementarySchool", "MiddleOrJuniorSchool", "HighSchool", "HighSchoolDistrict"]])

                        Percent  Counts
ElementarySchool      88.024982  481178
MiddleOrJuniorSchool  88.020774  481155
HighSchool            84.291432  460769
HighSchoolDistrict    29.999561  163989
                        Percent   Count
ElementarySchool      86.684775  358699
MiddleOrJuniorSchool  86.591251  358312
HighSchool            82.590014  341755
HighSchoolDistrict    27.425767  113487


In [447]:
# It could be possible to late fill in these unknown values from a school district database.
# Though high missing values, keep for now and fill with "Unknown"

listings[["ElementarySchool", "MiddleOrJuniorSchool",
          "HighSchool", "HighSchoolDistrict"]] = listings[["ElementarySchool","MiddleOrJuniorSchool",
                                                           "HighSchool", "HighSchoolDistrict"]].fillna("Unknown")

sold[["ElementarySchool", "MiddleOrJuniorSchool",
          "HighSchool", "HighSchoolDistrict"]] = sold[["ElementarySchool","MiddleOrJuniorSchool",
                                                           "HighSchool", "HighSchoolDistrict"]].fillna("Unknown")

In [448]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 49 columns
Shape of Sold Dataset: 411631 rows, 55 columns


### Property Qualities (Boolean)

In [449]:
# Check: AttachedGarageYN, FireplaceYN, NewConstructionYN (listings)
# Check: AttachedGarageYN, FireplaceYN, NewConstructionYN, ViewYN, WaterfrontYN, BasementYN, PoolPrivateYN (sold)

print(listings_na.loc[["AttachedGarageYN", "FireplaceYN", "NewConstructionYN"]])
print(sold_na.loc[["AttachedGarageYN", "FireplaceYN", "NewConstructionYN", "ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN"]])

                     Percent  Counts
AttachedGarageYN   17.421950   95235
FireplaceYN         0.102444     560
NewConstructionYN   6.871641   37563
                     Percent   Count
AttachedGarageYN   15.264973   63166
FireplaceYN         0.079266     328
NewConstructionYN   7.420788   30707
ViewYN              8.517945   35247
WaterfrontYN       99.937409  413538
BasementYN         98.043727  405702
PoolPrivateYN       8.609052   35624


In [450]:
# Simply flag as "unknown" for all missing boolean features

# Listings Dataset
listings["AttachedGarageYN"] = listings["AttachedGarageYN"].fillna("Unknown")
listings["FireplaceYN"] = listings["FireplaceYN"].fillna("Unknown")
listings["NewConstructionYN"] = listings["NewConstructionYN"].fillna("Unknown")

sold["AttachedGarageYN"] = sold["AttachedGarageYN"].fillna("Unknown")
sold["FireplaceYN"] = sold["FireplaceYN"].fillna("Unknown")
sold["NewConstructionYN"] = sold["NewConstructionYN"].fillna("Unknown")
sold["ViewYN"] = sold["ViewYN"].fillna("Unknown")
sold["WaterfrontYN"] = sold["WaterfrontYN"].fillna("Unknown")
sold["BasementYN"] = sold["BasementYN"].fillna("Unknown")
sold["PoolPrivateYN"] = sold["PoolPrivateYN"].fillna("Unknown")

In [451]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 49 columns
Shape of Sold Dataset: 411631 rows, 55 columns


### Property Qualities (Numeric)

In [452]:
# Check LivingArea, LotSizeAcres, LotSizeArea, LotSizeSquareFeet, Stories, Levels,
# ParkingTotal, GarageSpaces, BathroomsTotalInteger, BedroomsTotal, MainLevelBedrooms + Flooring (sold)

# Every feature here needs to be filled

print(listings_na.loc[["LivingArea", "LotSizeAcres", "LotSizeArea", "LotSizeSquareFeet", "Stories", "Levels",
                       "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms"]])
print(sold_na.loc[["LivingArea", "LotSizeAcres", "LotSizeArea", "LotSizeSquareFeet", "Stories", "Levels",
                  "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Flooring"]])

                         Percent  Counts
LivingArea              0.100249     548
LotSizeAcres            8.233968   45010
LotSizeArea             8.078289   44159
LotSizeSquareFeet       8.110120   44333
Stories                17.616046   96296
Levels                 10.952952   59873
ParkingTotal            0.003659      20
GarageSpaces            5.497788   30053
BathroomsTotalInteger   0.010062      55
BedroomsTotal           0.012074      66
MainLevelBedrooms      45.743252  250050
                         Percent   Count
LivingArea              0.056549     234
LotSizeAcres            7.860376   32526
LotSizeArea             7.732777   31998
LotSizeSquareFeet       7.763952   32127
Stories                15.403447   63739
Levels                  9.639509   39888
ParkingTotal            0.103916     430
GarageSpaces            4.272143   17678
BathroomsTotalInteger   0.016675      69
BedroomsTotal           0.002658      11
MainLevelBedrooms      41.957530  173619
Flooring        

In [453]:
# Fill null of Flooring with "Unknown"

sold["Flooring"] = sold["Flooring"].fillna("Unknown")

In [454]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 49 columns
Shape of Sold Dataset: 411631 rows, 55 columns


#### Stories and Levels

In [455]:
# Stories and Levels seem to encode similar data. Check the difference

print(listings["Levels"].value_counts())
print("-----")
print(listings["Stories"].value_counts())

Levels
One                               137564
Two                                92482
ThreeOrMore                        11395
MultiSplit                          2937
One,Two                              573
Two,MultiSplit                       343
Two,ThreeOrMore                      164
ThreeOrMore,MultiSplit               163
Two,One                              156
One,MultiSplit                       115
One,ThreeOrMore                       46
One,Two,ThreeOrMore                   28
MultiSplit,One                        16
One,Two,MultiSplit                    10
Two,ThreeOrMore,MultiSplit             5
One,Two,ThreeOrMore,MultiSplit         3
ThreeOrMore,One                        3
One,ThreeOrMore,MultiSplit             1
Name: count, dtype: int64
-----
Stories
1.0    139044
2.0     92755
Name: count, dtype: int64


In [456]:
# Check how Stories and Levels interact

listings[(listings["Levels"] != "One") & (listings["Levels"] != "Two")][["Stories", "Levels"]].value_counts()

Stories  Levels                        
1.0      One,Two                           573
2.0      Two,MultiSplit                    342
         Two,ThreeOrMore                   163
1.0      Two,One                           156
         One,MultiSplit                    115
         One,ThreeOrMore                    46
         One,Two,ThreeOrMore                28
         MultiSplit,One                     16
         MultiSplit                         10
         One,Two,MultiSplit                 10
         ThreeOrMore                         9
2.0      Two,ThreeOrMore,MultiSplit          5
1.0      ThreeOrMore,One                     3
         One,Two,ThreeOrMore,MultiSplit      3
         Two,MultiSplit                      1
         Two,ThreeOrMore                     1
         One,ThreeOrMore,MultiSplit          1
Name: count, dtype: int64

In [457]:
# What values in Levels are showing up as Null in Stories

print(listings[listings["Stories"].isna() == True]["Levels"].value_counts(dropna=False))
print(sold[sold["Stories"].isna() == True]["Levels"].value_counts(dropna=False))

Levels
NaN                       22742
ThreeOrMore               11386
MultiSplit                 2927
ThreeOrMore,MultiSplit      163
One                           1
Name: count, dtype: int64
Levels
NaN                       39605
ThreeOrMore               16986
MultiSplit                 6554
ThreeOrMore,MultiSplit      406
Name: count, dtype: int64


In [458]:
# Impute Stories by filling in these null values.
# Multisplit means a loft, so represent as a half value.

listings.loc[(listings["Stories"].isna()) & (listings["Levels"] == "ThreeOrMore"), "Stories"] = 3.0
listings.loc[(listings["Stories"].isna()) & (listings["Levels"] == "MultiSplit"), "Stories"] = 1.5
listings.loc[(listings["Stories"].isna()) & (listings["Levels"] == "ThreeOrMore,MultiSplit"), "Stories"]= 3.5
listings.loc[(listings["Stories"].isna()) & (listings["Levels"] == "One"), "Stories"] = 1

sold.loc[(sold["Stories"].isna()) & (sold["Levels"] == "ThreeOrMore"), "Stories"] = 3.0
sold.loc[(sold["Stories"].isna()) & (sold["Levels"] == "MultiSplit"), "Stories"] = 1.5
sold.loc[(sold["Stories"].isna()) & (sold["Levels"] == "ThreeOrMore,MultiSplit"), "Stories"] = 3.5

print(listings["Stories"].value_counts(dropna=False))
print(sold["Stories"].value_counts(dropna=False))

Stories
1.0    139045
2.0     92755
NaN     22742
3.0     11386
1.5      2927
3.5       163
Name: count, dtype: int64
Stories
1.0    221425
2.0    126655
NaN     39605
3.0     16986
1.5      6554
3.5       406
Name: count, dtype: int64


In [459]:
# Calculate LivingArea median for each story option

# Listings Dataset
one_story_median_L = listings[listings["Stories"] == 1.0 ]["LivingArea"].median()
one_half_story_median_L = listings[listings["Stories"] == 1.5 ]["LivingArea"].median()
two_story_median_L = listings[listings["Stories"] == 2.0 ]["LivingArea"].median()
three_story_median_L = listings[listings["Stories"] == 3.0 ]["LivingArea"].median()
three_half_story_median_L = listings[listings["Stories"] == 3.5 ]["LivingArea"].median()

print(one_story_median_L, one_half_story_median_L, two_story_median_L, three_story_median_L, three_half_story_median_L)

# Sold Dataset
one_story_median_S = sold[sold["Stories"] == 1.0 ]["LivingArea"].median()
one_half_story_median_S = sold[sold["Stories"] == 1.5 ]["LivingArea"].median()
two_story_median_S = sold[sold["Stories"] == 2.0 ]["LivingArea"].median()
three_story_median_S = sold[sold["Stories"] == 3.0 ]["LivingArea"].median()
three_half_story_median_S = sold[sold["Stories"] == 3.5 ]["LivingArea"].median()

print(one_story_median_S, one_half_story_median_S, two_story_median_S, three_story_median_S, three_half_story_median_S)

1486.0 2040.0 2148.0 1877.0 2112.0
1441.0 1934.0 2112.0 1786.0 2019.0


In [460]:
# Impute remaining missing stories based on LivingArea (listings dataset)

midpoint_1_L = (one_story_median_L + one_half_story_median_L) / 2
midpoint_2_L = (one_half_story_median_L + two_story_median_L) / 2
midpoint_3_L = (two_story_median_L + three_story_median_L) / 2
midpoint_4_L = (three_story_median_L + three_half_story_median_L) / 2

# [-, midpoint_1) = 1 story
# [midpoint_1, midpoint_2) = 1.5 stories
# [midpoint_2, midpoint_3) = 2 stories
# [midpoint_3, midpoint_4) = 3 stories
# [midpint_4, +] = 3.5 stories

listings.loc[(listings["Stories"].isna()) & 
                    (listings["LivingArea"] < midpoint_1_L),
                    "Stories"] = 1.0

listings.loc[(listings["Stories"].isna()) & 
                    (listings["LivingArea"] >= midpoint_1_L) &
                    (listings["LivingArea"] < midpoint_2_L),
                    "Stories"] = 1.5

listings.loc[(listings["Stories"].isna()) & 
                    (listings["LivingArea"] >= midpoint_2_L) &
                    (listings["LivingArea"] < midpoint_3_L),
                    "Stories"]= 2.0

listings.loc[(listings["Stories"].isna()) & 
                    (listings["LivingArea"] >= midpoint_3_L) &
                    (listings["LivingArea"] < midpoint_4_L),
                    "Stories"] = 3.0

listings.loc[(listings["Stories"].isna()) & 
                    (listings["LivingArea"] >= midpoint_4_L),
                    "Stories"] = 3.5

In [461]:
# Impute remaining missing stories based on LivingArea (sold dataset)

midpoint_1_S = (one_story_median_S + one_half_story_median_S) / 2
midpoint_2_S = (one_half_story_median_S + two_story_median_S) / 2
midpoint_3_S = (two_story_median_S + three_story_median_S) / 2
midpoint_4_S = (three_story_median_S + three_half_story_median_S) / 2

# [-, midpoint_1) = 1 story
# [midpoint_1, midpoint_2) = 1.5 stories
# [midpoint_2, midpoint_3) = 2 stories
# [midpoint_3, midpoint_4) = 3 stories
# [midpint_4, +] = 3.5 stories

sold.loc[(sold["Stories"].isna()) & 
        (sold["LivingArea"] < midpoint_1_S),
        "Stories"] = 1.0

sold.loc[(sold["Stories"].isna()) & 
        (sold["LivingArea"] >= midpoint_1_S) &
        (sold["LivingArea"] < midpoint_2_S),
        "Stories"] = 1.5

sold.loc[(sold["Stories"].isna()) & 
        (sold["LivingArea"] >= midpoint_2_S) &
        (sold["LivingArea"] < midpoint_3_S),
        "Stories"]= 2.0

sold.loc[(listings["Stories"].isna()) & 
        (sold["LivingArea"] >= midpoint_3_S) &
        (sold["LivingArea"] < midpoint_4_S),
        "Stories"] = 3.0

sold.loc[(sold["Stories"].isna()) & 
        (sold["LivingArea"] >= midpoint_4_S),
        "Stories"] = 3.5

In [462]:
# Impute the remaining missing Stories value with the median of Stories

print(listings["Stories"].isna().sum())
print(sold["Stories"].isna().sum())

listings["Stories"] = listings["Stories"].fillna(listings["Stories"].median())
sold["Stories"] = sold["Stories"].fillna(sold["Stories"].median())

38
23


In [463]:
# Drop Levels column, as its information is held in Stories now

listings = listings.drop(columns = ["Levels"])
sold = sold.drop(columns = ["Levels"])

In [464]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 48 columns
Shape of Sold Dataset: 411631 rows, 54 columns


#### Bathrooms and Bedrooms

In [465]:
# Get median bathrooms from each story

one_story_bath_L = listings[listings["Stories"] == 1.0 ]["BathroomsTotalInteger"].median()
one_half_story_bath_L = listings[listings["Stories"] == 1.5 ]["BathroomsTotalInteger"].median()
two_story_bath_L = listings[listings["Stories"] == 2.0 ]["BathroomsTotalInteger"].median()
three_story_bath_L = listings[listings["Stories"] == 3.0 ]["BathroomsTotalInteger"].median()
three_half_story_bath_L = listings[listings["Stories"] == 3.5 ]["BathroomsTotalInteger"].median()

one_story_bath_S = sold[sold["Stories"] == 1.0 ]["BathroomsTotalInteger"].median()
one_half_story_bath_S = sold[sold["Stories"] == 1.5 ]["BathroomsTotalInteger"].median()
two_story_bath_S = sold[sold["Stories"] == 2.0 ]["BathroomsTotalInteger"].median()
three_story_bath_S = sold[sold["Stories"] == 3.0 ]["BathroomsTotalInteger"].median()
three_half_story_bath_S = sold[sold["Stories"] == 3.5 ]["BathroomsTotalInteger"].median()

In [466]:
# Impute BathroomsTotalInteger with median bathrooms of each story

listings.loc[(listings["BathroomsTotalInteger"].isna()) & (listings["Stories"] == 1.0), "BathroomsTotalInteger"] = one_story_bath_L
listings.loc[(listings["BathroomsTotalInteger"].isna()) & (listings["Stories"] == 1.5), "BathroomsTotalInteger"] = one_half_story_bath_L
listings.loc[(listings["BathroomsTotalInteger"].isna()) & (listings["Stories"] == 2.0), "BathroomsTotalInteger"] = two_story_bath_L
listings.loc[(listings["BathroomsTotalInteger"].isna()) & (listings["Stories"] == 3.0), "BathroomsTotalInteger"] = three_story_bath_L
listings.loc[(listings["BathroomsTotalInteger"].isna()) & (listings["Stories"] == 3.5), "BathroomsTotalInteger"] = three_half_story_bath_L

sold.loc[(sold["BathroomsTotalInteger"].isna()) & (sold["Stories"] == 1.0), "BathroomsTotalInteger"] = one_story_bath_S
sold.loc[(sold["BathroomsTotalInteger"].isna()) & (sold["Stories"] == 1.5), "BathroomsTotalInteger"] = one_half_story_bath_S
sold.loc[(sold["BathroomsTotalInteger"].isna()) & (sold["Stories"] == 2.0), "BathroomsTotalInteger"] = two_story_bath_S
sold.loc[(sold["BathroomsTotalInteger"].isna()) & (sold["Stories"] == 3.0), "BathroomsTotalInteger"] = three_story_bath_S
sold.loc[(sold["BathroomsTotalInteger"].isna()) & (sold["Stories"] == 3.5), "BathroomsTotalInteger"] = three_half_story_bath_S

In [467]:
# Do the same for BedroomsTotal
# Get median bedrooms from each story

one_story_bed_L = listings[listings["Stories"] == 1.0 ]["BedroomsTotal"].median()
one_half_story_bed_L = listings[listings["Stories"] == 1.5 ]["BedroomsTotal"].median()
two_story_bed_L = listings[listings["Stories"] == 2.0 ]["BedroomsTotal"].median()
three_story_bed_L = listings[listings["Stories"] == 3.0 ]["BedroomsTotal"].median()
three_half_story_bed_L = listings[listings["Stories"] == 3.5 ]["BedroomsTotal"].median()

one_story_bed_S = sold[sold["Stories"] == 1.0 ]["BedroomsTotal"].median()
one_half_story_bed_S = sold[sold["Stories"] == 1.5 ]["BedroomsTotal"].median()
two_story_bed_S = sold[sold["Stories"] == 2.0 ]["BedroomsTotal"].median()
three_story_bed_S = sold[sold["Stories"] == 3.0 ]["BedroomsTotal"].median()
three_half_story_bed_S = sold[sold["Stories"] == 3.5 ]["BedroomsTotal"].median()

In [468]:
# Impute BedroomsTotal with median bathrooms of each story

listings.loc[(listings["BedroomsTotal"].isna()) & (listings["Stories"] == 1.0), "BedroomsTotal"] = one_story_bed_L
listings.loc[(listings["BedroomsTotal"].isna()) & (listings["Stories"] == 1.5), "BedroomsTotal"] = one_half_story_bed_L
listings.loc[(listings["BedroomsTotal"].isna()) & (listings["Stories"] == 2.0), "BedroomsTotal"] = two_story_bed_L
listings.loc[(listings["BedroomsTotal"].isna()) & (listings["Stories"] == 3.0), "BedroomsTotal"] = three_story_bed_L
listings.loc[(listings["BedroomsTotal"].isna()) & (listings["Stories"] == 3.5), "BedroomsTotal"] = three_half_story_bed_L

sold.loc[(sold["BedroomsTotal"].isna()) & (sold["Stories"] == 1.0), "BedroomsTotal"] = one_story_bed_S
sold.loc[(sold["BedroomsTotal"].isna()) & (sold["Stories"] == 1.5), "BedroomsTotal"] = one_half_story_bed_S
sold.loc[(sold["BedroomsTotal"].isna()) & (sold["Stories"] == 2.0), "BedroomsTotal"] = two_story_bed_S
sold.loc[(sold["BedroomsTotal"].isna()) & (sold["Stories"] == 3.0), "BedroomsTotal"] = three_story_bed_S
sold.loc[(sold["BedroomsTotal"].isna()) & (sold["Stories"] == 3.5), "BedroomsTotal"] = three_half_story_bed_S

In [469]:
# If only one story, impute MainLevelBedrooms as same as BedroomsTotal.
# Otherwise, impute average of non one story properties

# If there is one story, fill MainLevelBedrooms with BedroomsTotal.
listings["MainLevelBedrooms"] = np.where((listings["Stories"] == 1.0) &
                                        listings["MainLevelBedrooms"].isna() == True,
                                        listings["BedroomsTotal"], listings["MainLevelBedrooms"])

plus_one_story_bed_median_L = listings[listings["Stories"] > 1.0]["MainLevelBedrooms"].median()
listings["MainLevelBedrooms"] = listings["MainLevelBedrooms"].fillna(plus_one_story_bed_median_L)


# Repeat for Sold dataset
sold["MainLevelBedrooms"] = np.where((sold["Stories"] == 1.0) &
                                    sold["MainLevelBedrooms"].isna() == True,
                                    sold["BedroomsTotal"], sold["MainLevelBedrooms"])

plus_one_story_bed_median_S = sold[sold["Stories"] > 1.0]["MainLevelBedrooms"].median()
sold["MainLevelBedrooms"] = sold["MainLevelBedrooms"].fillna(plus_one_story_bed_median_S)

In [470]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 48 columns
Shape of Sold Dataset: 411631 rows, 54 columns


#### Parking

In [471]:
# For all the ParkingTotal spaces that are null, the GarageSpaces column has values

print(listings[listings["ParkingTotal"].isna() == True]["GarageSpaces"].value_counts(dropna=False))
print(sold[sold["ParkingTotal"].isna() == True]["GarageSpaces"].value_counts(dropna=False))

GarageSpaces
1.0    3
2.0    2
0.0    2
3.0    1
Name: count, dtype: int64
GarageSpaces
2.0    259
1.0     65
0.0     43
3.0     28
4.0      6
5.0      2
Name: count, dtype: int64


In [472]:
# Impute missing ParkingTotal values with GarageSpaces

listings["ParkingTotal"] = listings["ParkingTotal"].fillna(listings["GarageSpaces"])
sold["ParkingTotal"] = sold["ParkingTotal"].fillna(sold["GarageSpaces"])

In [473]:
# Now, impute values for GarageSpaces by using AttachedGarageYN.
# AttachedGarageYN = false does not necessarily mean no garage exists, just that it's not attached.
# Take median GarageSpaces for each value of AttachedGarageYN

garage_true_L = listings[listings["AttachedGarageYN"] == True]["GarageSpaces"].median()
garage_false_L = listings[listings["AttachedGarageYN"] == False]["GarageSpaces"].median()
garage_idk_L = listings[listings["AttachedGarageYN"] == "Unknown"]["GarageSpaces"].median()

garage_true_S = sold[sold["AttachedGarageYN"] == True]["GarageSpaces"].median()
garage_false_S = sold[sold["AttachedGarageYN"] == False]["GarageSpaces"].median()
garage_idk_S = sold[sold["AttachedGarageYN"] == "Unknown"]["GarageSpaces"].median()

In [474]:
# Impute GarageSpaces with the medians created using AttachedGarageYN

listings.loc[(listings["GarageSpaces"].isna()) & (listings["AttachedGarageYN"] == True), "GarageSpaces"] = garage_true_L
listings.loc[(listings["GarageSpaces"].isna()) & (listings["AttachedGarageYN"] == False), "GarageSpaces"] = garage_false_L
listings.loc[(listings["GarageSpaces"].isna()) & (listings["AttachedGarageYN"] == "Unknown"), "GarageSpaces"]= garage_idk_L

sold.loc[(sold["GarageSpaces"].isna()) & (sold["AttachedGarageYN"] == True), "GarageSpaces"] = garage_true_S
sold.loc[(sold["GarageSpaces"].isna()) & (sold["AttachedGarageYN"] == False), "GarageSpaces"] = garage_false_S
sold.loc[(sold["GarageSpaces"].isna()) & (sold["AttachedGarageYN"] == "Unknown"), "GarageSpaces"]= garage_idk_S

In [475]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 48 columns
Shape of Sold Dataset: 411631 rows, 54 columns


#### LotSize Units

In [476]:
# Check differences in units

### Notes
# LotSizeArea uses ambigious units, seemingly a combination of LotSizeAcres or LotSizeSquareFeet, not super good to use
# This leaves LotSizeAcres and LotSizeSquareFeet. LotSizeSquareFeet appears better since its values are more precise

listings[["LotSizeAcres", "LotSizeArea", "LotSizeSquareFeet"]]

,LotSizeAcres,LotSizeArea,LotSizeSquareFeet
1,0.1217,5300.00,5300.0
3,1.3368,58232.00,58232.0
4,0.0517,2250.00,2250.0
9,NaN,NaN,NaN
10,2.1000,91476.00,91476.0
...,...,...,...
547149,0.1926,8389.00,8389.0
547150,0.1263,5500.00,5500.0
547152,0.0764,3330.00,3330.0
547154,0.2066,9000.00,9000.0


In [477]:
# Since LotSizeSquareFeet is the best LotSize column, investigate if other features can be used to fill its null values
# Where LotSizeSquareFeet is null and the other values exist, both values seem to be in square feet

listings[(listings["LotSizeSquareFeet"].isna() == True) &
        (listings["LotSizeAcres"].isna() == False) &
        (listings["LotSizeArea"].isna() == False)][["LotSizeAcres", "LotSizeArea"]].head(8)

,LotSizeAcres,LotSizeArea
12628,127205.0,127205.0
19686,57092.0,57092.0
27894,406432.0,406432.0
47165,87120.0,87120.0
90223,0.0,0.0
90874,145160.0,145160.0
93040,0.0,0.0
93418,0.0,0.0


In [478]:
# Fill missing LotSizeSquareFeet values with LotSizeArea

listings["LotSizeSquareFeet"] = listings["LotSizeSquareFeet"].fillna(listings["LotSizeArea"])
sold["LotSizeSquareFeet"] = sold["LotSizeSquareFeet"].fillna(sold["LotSizeArea"])

In [479]:
# Having multiple LotSize columns in redundant. Keep LotSizeSquareFeet and drop LotSizeAcres and LotSizeArea

listings = listings.drop(columns = ["LotSizeAcres", "LotSizeArea"])
sold = sold.drop(columns = ["LotSizeAcres", "LotSizeArea"])

In [480]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 46 columns
Shape of Sold Dataset: 411631 rows, 52 columns


#### LotSizeSquareFeet and LivingArea

In [481]:
# Calculate the median LivingArea and LotSizeSquareFeet for each number of stories

# LivingArea (listings)
one_story_living_L = listings[listings["Stories"] == 1.0]["LivingArea"].median()
half_story_living_L = listings[listings["Stories"] == 1.5]["LivingArea"].median()
two_story_living_L = listings[listings["Stories"] == 2.0]["LivingArea"].median()
three_story_living_L = listings[listings["Stories"] == 3.0]["LivingArea"].median()
three_half_story_living_L = listings[listings["Stories"] == 3.5]["LivingArea"].median()

# LotSizeSquareFeet (listings)
one_story_feet_L = listings[listings["Stories"] == 1.0]["LotSizeSquareFeet"].median()
half_story_feet_L = listings[listings["Stories"] == 1.5]["LotSizeSquareFeet"].median()
two_story_feet_L = listings[listings["Stories"] == 2.0]["LotSizeSquareFeet"].median()
three_story_feet_L = listings[listings["Stories"] == 3.0]["LotSizeSquareFeet"].median()
three_half_story_feet_L = listings[listings["Stories"] == 3.5]["LotSizeSquareFeet"].median()

# LivingArea (sold)
one_story_living_S = sold[sold["Stories"] == 1.0]["LivingArea"].median()
half_story_living_S = sold[sold["Stories"] == 1.5]["LivingArea"].median()
two_story_living_S = sold[sold["Stories"] == 2.0]["LivingArea"].median()
three_story_living_S = sold[sold["Stories"] == 3.0]["LivingArea"].median()
three_half_story_living_S = sold[sold["Stories"] == 3.5]["LivingArea"].median()

# LotSizeSquareFeet (sold)
one_story_feet_S = sold[sold["Stories"] == 1.0]["LotSizeSquareFeet"].median()
half_story_feet_S = sold[sold["Stories"] == 1.5]["LotSizeSquareFeet"].median()
two_story_feet_S = sold[sold["Stories"] == 2.0]["LotSizeSquareFeet"].median()
three_story_feet_S = sold[sold["Stories"] == 3.0]["LotSizeSquareFeet"].median()
three_half_story_feet_S = sold[sold["Stories"] == 3.5]["LotSizeSquareFeet"].median()

In [482]:
# Impute LivingArea from previous medians

listings.loc[(listings["LivingArea"].isna()) & (listings["Stories"] == 1.0), "LivingArea"] = one_story_living_L
listings.loc[(listings["LivingArea"].isna()) & (listings["Stories"] == 1.5), "LivingArea"] = half_story_living_L
listings.loc[(listings["LivingArea"].isna()) & (listings["Stories"] == 2.0), "LivingArea"] = two_story_living_L
listings.loc[(listings["LivingArea"].isna()) & (listings["Stories"] == 3.0), "LivingArea"] = three_story_living_L
listings.loc[(listings["LivingArea"].isna()) & (listings["Stories"] == 3.5), "LivingArea"] = three_half_story_living_L

sold.loc[(sold["LivingArea"].isna()) & (sold["Stories"] == 1.0), "LivingArea"] = one_story_living_S
sold.loc[(sold["LivingArea"].isna()) & (sold["Stories"] == 1.5), "LivingArea"] = half_story_living_S
sold.loc[(sold["LivingArea"].isna()) & (sold["Stories"] == 2.0), "LivingArea"] = two_story_living_S
sold.loc[(sold["LivingArea"].isna()) & (sold["Stories"] == 3.0), "LivingArea"] = three_story_living_S
sold.loc[(sold["LivingArea"].isna()) & (sold["Stories"] == 3.5), "LivingArea"] = three_half_story_living_S

In [483]:
# Impute LotSizeSquareFeet from previous medians

listings.loc[(listings["LotSizeSquareFeet"].isna()) & (listings["Stories"] == 1.0), "LotSizeSquareFeet"] = one_story_feet_L
listings.loc[(listings["LotSizeSquareFeet"].isna()) & (listings["Stories"] == 1.5), "LotSizeSquareFeet"] = half_story_feet_L
listings.loc[(listings["LotSizeSquareFeet"].isna()) & (listings["Stories"] == 2.0), "LotSizeSquareFeet"] = two_story_feet_L
listings.loc[(listings["LotSizeSquareFeet"].isna()) & (listings["Stories"] == 3.0), "LotSizeSquareFeet"] = three_story_feet_L
listings.loc[(listings["LotSizeSquareFeet"].isna()) & (listings["Stories"] == 3.5), "LotSizeSquareFeet"] = three_half_story_feet_L

sold.loc[(sold["LotSizeSquareFeet"].isna()) & (sold["Stories"] == 1.0), "LotSizeSquareFeet"] = one_story_feet_S
sold.loc[(sold["LotSizeSquareFeet"].isna()) & (sold["Stories"] == 1.5), "LotSizeSquareFeet"] = half_story_feet_S
sold.loc[(sold["LotSizeSquareFeet"].isna()) & (sold["Stories"] == 2.0), "LotSizeSquareFeet"] = two_story_feet_S
sold.loc[(sold["LotSizeSquareFeet"].isna()) & (sold["Stories"] == 3.0), "LotSizeSquareFeet"] = three_story_feet_S
sold.loc[(sold["LotSizeSquareFeet"].isna()) & (sold["Stories"] == 3.5), "LotSizeSquareFeet"] = three_half_story_feet_S

In [484]:
print(f"Shape of Listings Dataset: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Shape of Sold Dataset: {sold.shape[0]} rows, {sold.shape[1]} columns")

Shape of Listings Dataset: 269018 rows, 46 columns
Shape of Sold Dataset: 411631 rows, 52 columns


## Handling Summary

In [485]:
listings.isna().sum()

OriginalListPrice            0
ListingKey                   0
CloseDate                    0
ClosePrice                   0
Latitude                     0
Longitude                    0
UnparsedAddress              0
LivingArea                   0
ListPrice                    0
DaysOnMarket                 0
ListOfficeName               0
BuyerOfficeName              0
CoListOfficeName             0
ListAgentFullName            0
MLSAreaMajor                 0
CountyOrParish               0
MlsStatus                    0
ElementarySchool             0
AttachedGarageYN             0
ParkingTotal                 0
PropertySubType              0
YearBuilt                    0
ListingId                    0
BathroomsTotalInteger        0
City                         0
BedroomsTotal                0
ContractStatusChangeDate     0
PurchaseContractDate         0
ListingContractDate          0
StateOrProvince              0
MiddleOrJuniorSchool         0
FireplaceYN                  0
Stories 

In [486]:
sold.isna().sum()

Flooring                     0
ViewYN                       0
WaterfrontYN                 0
BasementYN                   0
PoolPrivateYN                0
OriginalListPrice            0
ListingKey                   0
CloseDate                    0
ClosePrice                   0
Latitude                     0
Longitude                    0
UnparsedAddress              0
LivingArea                   0
ListPrice                    0
DaysOnMarket                 0
ListOfficeName               0
BuyerOfficeName              0
CoListOfficeName             0
ListAgentFullName            0
BuyerAgentMlsId              0
MLSAreaMajor                 0
CountyOrParish               0
ElementarySchool             0
AttachedGarageYN             0
ParkingTotal                 0
PropertySubType              0
YearBuilt                    0
ListingId                    0
BathroomsTotalInteger        0
City                         0
BedroomsTotal                0
ContractStatusChangeDate     0
Purchase


| Dropped | Filled With "Unknown" | Combined with Another Column | No Change | Imputed 
| -------- | -------- | -------- | -------- | -------- |
| ListingKeyNumeric | ListAgentFullName | CoListAgentFirstName | MlsStatus (listings) | OriginalListPrice
| ListAgentFirstName | CoListAgentFullName | CoListAgentLastName| ListingKey | StateOrProvince
| ListAgentLastName | BuyerAgentFullName | BuyerAgentFirstName | ListingId | Longitude
| BuyerAgencyCompensationType | BuyerAgentMlsId (sold) | BuyerAgentLastName | ListPrice | Latitude
| BuyerAgencyCompensation | PropertySubType | | rate_30yr_fixed | City
| AssociationFee | ClosePrice | | ListOfficeName | Stories
| AssociationFeeFrequency | BuyerOfficeName | | DaysOnMarket | BathroomsTotalInteger
| ListAgentEmail | CoListOfficeName | | ListingContractDate | BedroomsTotal
| CoBuyerAgentFirstName | CloseDate | | year_month | MainLevelBedrooms
| BuyerAgentMlsId (listings) | YearBuilt | | CountyOrParish | ParkingTotal
| BuyerOfficeAOR | ContractStatusChangeDate | | PostalCode | GarageSpaces
| ListAgentAOR | PurchaseContractDate | | | LotSizeSquareFeet
| BuyerAgentAOR | MLSAreaMajor | | | LivingArea
| BuilderName | ElementarySchool | | | 
| OriginatingSystemName | MiddleOrJuniorSchool | | |
| OriginatingSystemSubName | HighSchool | | |
| latfilled | HighSchoolDistrict | | |
| lonfilled | ViewYN | | |
| PropertyType | WaterfrontYN | | |
| MlsStatus (sold) | BasementYN | | |
| StreetNumberNumeric | PoolPrivateYN | | |
| UnparsedAddress | AttachedGarageYN | | | 
| SubdivisionName | FireplaceYN| | |
| Levels | NewConstructionYN | | |
| LotSizeAcres | Flooring | | |
| LotSizeArea | | | |

#### More Detailed Descriptions:
* **ListingKeyNumeric**: dropped
* **ListAgentFirstName**: dropped
* **ListAgentLastName**: dropped
* **ListAgentFullName**: filled with "Unknown"
* **CoListAgentFirstName**: combined into ListAgentFullName, dropped
* **CoListAgentLastName**: combined into ListAgentFullName, dropped
* **CoListAgentFullName**: custom column, filled with "Unknown"
* **BuyerAgentFirstName**: combined into BuyerAgentFullName, dropped
* **BuyerAgentLastName**: combined into BuyerAgentFullName, dropped
* **BuyerAgentFullName**: custom column, filled with "Unknown"
* **BuyerAgencyCompensationType**: dropped
* **BuyerAgencyCompensation**: dropped
* **AssociationFee**: dropped
* **AssociationFeeFrequency**: dropped
* **ListAgentEmail**: dropped
* **CoBuyerAgentFirstName**: dropped
* **BuyerAgentMlsId**: dropped (listings), filled with "Unknown" (sold)
* **BuyerOfficeAOR**: dropped
* **ListAgentAOR**: dropped
* **BuyerAgentAOR**: dropped
* **BuilderName**: dropped
* **OriginatingSystemName**: dropped
* **OriginatingSystemSubName**: dropped
* **latfilled**: dropped
* **lonfilled**: dropped
* **PropertyType**: dropped
* **MlsStatus**: no change (listings), dropped (sold)
* **ListingKey**: no change
* **ListingId**: no change
* **PropertySubType**: filled nulls with "Unknown"
* **ListPrice**: no change
* **rate_30yr_fixed**: no change
* **OriginalListPrice**: nulls filled with ListPrice
* **ClosePrice**: filled with "N/A" or "Unknown" based on MlsStatus (listings), null rows removed (sold)
* **ListOfficeName**: no change
* **BuyerOfficeName**: filled with "N/A" or "Unknown" based on MlsStatus (listings), filled with "Unknown" (sold)
* **CoListOfficeName**: filled with "Unknown"
* **CloseDate**: filled with "N/A" or "Unknown" based on MlsStatus (listings), no change (sold)
* **DaysOnMarket**: no change
* **YearBuilt**: filled with "Unknown"
* **ContractStatusChangeDate**: filled with "Unknown"
* **PurchaseContractDate**: filled with "Unknown"
* **ListingContractDate**: no change (listings), null rows removed (sold)
* **year_month**: no change
* **CountyOrParish**: no change
* **StateOrProvince**: imputed with uszipcodes dataset (listings), no change (sold)
* **StreetNumberNumeric**: dropped
* **UnparsedAddress**: dropped
* **PostalCode**: null rows removed
* **SubdivisionName**: dropped
* **MLSAreaMajor**: filled with "Unknown"
* **Longitude**: imputed with uszipcodes dataset
* **Latitude**: imputed with uszipcodes dataset
* **City**: imputed with uszipcodes dataset
* **ElementarySchool**: filled with "Unknown"
* **MiddleOrJuniorSchool**: filled with "Unknown"
* **HighSchool**: filled with "Unknown"
* **HighSchoolDistrict**: filled with "Unknown"
* **ViewYN**: filled with "Unknown"
* **WaterfrontYN**: filled with "Unknown"
* **BasementYN**: filled with "Unknown"
* **PoolPrivateYN**: filled with "Unknown"
* **AttachedGarageYN**: filled with "Unknown"
* **FireplaceYN**: filled with "Unknown"
* **NewConstructionYN**: filled with "Unknown"
* **Flooring**: filled with "Unknown"
* **Stories**: for rows with missing Stories but filled Levels, impute by using Levels as a reference.
For rows with missing Stories and Levels, use LivingArea to impute Stories
* **Levels**: dropped
* **BathroomsTotalInteger**: impute with median bathrooms from each story amount
* **BedroomsTotal**: impute with median bedrooms from each story amount
* **MainLevelBedrooms**: for properties with only one story, impute MainLevelBedrooms as BedroomsTotal. Otherwise, fill with median bedrooms for all properties with more than one story.
* **ParkingTotal**: impute with GarageSpaces
* **GarageSpaces**: impute with median GarageSpaces for each value of AttachedGarageYN (True/False/Unknown)
* **LotSizeSquareFeet**: fill with LotSizeArea, fill remaining values with median of LotSizeSquareFeet for each story
* **LotSizeAcres**: dropped
* **LotSizeArea**: dropped
* **LivingArea**: fill values with median of LivingArea for each story